# Cálculo de Volumen y Selección de Depósito GLP



In [1]:
import pandas as pd
import numpy as np

# 1. Parámetros y Constantes (Fuente: Proyecto/Anotaciones/calculo_volumen_deposito.md)
PCS_PROPANO = 13.95  # kWh/kg
DENSIDAD_LIQ = 506.0  # kg/m3 a 20°C
LLENADO_MAX = 0.85
RESERVA_MIN = 0.20
FRACCION_UTIL = LLENADO_MAX - RESERVA_MIN
AUTONOMIA_DIAS = 30  # Requerido por TAREA 3
TEMP_DISENO = -5.0   # IDAE Percentil 99,6% para León
PRESION_SERVICIO = 2.0 # bar (Condición de comprobación solicitada)

print(f"Fracción útil adoptada: {FRACCION_UTIL:.2f}")
print(f"Autonomía de diseño: {AUTONOMIA_DIAS} días")
print(f"Temperatura exterior de diseño: {TEMP_DISENO} ºC")
print(f"Presión de servicio para comprobación: {PRESION_SERVICIO} bar")

Fracción útil adoptada: 0.65
Autonomía de diseño: 30 días
Temperatura exterior de diseño: -5.0 ºC
Presión de servicio para comprobación: 2.0 bar


In [2]:
# 2. Datos de Consumidores (Fuente: Proyecto/Datos.md)
consumidores = [
    {"nombre": "Horno secado 1", "potencia_kw": 60, "horas_dia": 12},
    {"nombre": "Horno secado 2", "potencia_kw": 60, "horas_dia": 12},
    {"nombre": "Caldera vapor", "potencia_kw": 500, "horas_dia": 10},
    {"nombre": "Caldera agua caliente", "potencia_kw": 300, "horas_dia": 8},
    {"nombre": "Horno fusion", "potencia_kw": 700, "horas_dia": 4},
    {"nombre": "Horno decapado", "potencia_kw": 1000, "horas_dia": 6}
]

df_cons = pd.DataFrame(consumidores)
df_cons['energia_diaria_kwh'] = df_cons['potencia_kw'] * df_cons['horas_dia']

energia_total_dia = df_cons['energia_diaria_kwh'].sum()
potencia_max_simultanea = df_cons['potencia_kw'].sum()

print(f"Energía total diaria: {energia_total_dia} kWh/d")
print(f"Potencia máxima simultánea (S=1): {potencia_max_simultanea} kW")

Energía total diaria: 17640 kWh/d
Potencia máxima simultánea (S=1): 2620 kW


In [3]:
# 3. Cálculo de Consumo Másico y Volumétrico
m_dia = energia_total_dia / PCS_PROPANO
v_liq_dia = m_dia / DENSIDAD_LIQ  # m3/día

v_geom_min = (v_liq_dia * AUTONOMIA_DIAS) / FRACCION_UTIL

print(f"Consumo diario: {m_dia:.2f} kg/día")
print(f"Volumen líquido diario: {v_liq_dia:.3f} m3/día")
print(f"--- Volumen geométrico mínimo requerido: {v_geom_min:.2f} m3 ---")

Consumo diario: 1264.52 kg/día
Volumen líquido diario: 2.499 m3/día
--- Volumen geométrico mínimo requerido: 115.34 m3 ---


In [4]:
# 4. Selección del Depósito Comercial (Nueva Configuración por Distancias)
df_dep = pd.read_csv('datos/tabla_caracteristicas_secadores.csv')
df_vap_table = pd.read_csv('datos/caudal_vaporizacion.csv')
df_vapi_table = pd.read_csv('datos/deposito_vaporizador_interno.csv')

# Datos de la nueva selección
dep_grande_ref = 'LP46A-22'
dep_peque_ref = 'LP26A-22'
n_grande = 1
n_peque = 3

v_grande = df_dep[df_dep['Modelo Ref.'] == dep_grande_ref]['Capacidad nominal (litros)'].values[0]
v_peque = df_dep[df_dep['Modelo Ref.'] == dep_peque_ref]['Capacidad nominal (litros)'].values[0]
v_total = (v_grande * n_grande) + (v_peque * n_peque)

print(f"Configuración de Almacenamiento: {n_grande} x {dep_grande_ref} + {n_peque} x {dep_peque_ref}")
print(f"Capacidad total: {v_total} litros")

Configuración de Almacenamiento: 1 x LP46A-22 + 3 x LP26A-22
Capacidad total: 125100 litros


In [5]:
# 5. Verificación de Vaporización y Selección de Vaporizador Forzado
caudal_nec_kgh = potencia_max_simultanea / PCS_PROPANO
print(f"Demanda punta necesaria: {caudal_nec_kgh:.2f} kg/h")

# 5.1. Comprobación de Vaporización Natural (al 20% de llenado)
vol_grande_m3 = 46.2
vol_peque_m3 = 26.3

vap_grande = df_vap_table[(abs(df_vap_table['Volum. m3'] - vol_grande_m3) < 0.01) & (df_vap_table['Pres. bar'] == PRESION_SERVICIO)]
vap_peque = df_vap_table[(abs(df_vap_table['Volum. m3'] - vol_peque_m3) < 0.01) & (df_vap_table['Pres. bar'] == PRESION_SERVICIO)]

q_nat_grande = vap_grande['Caudal Aéreo -5°C'].values[0]
q_nat_peque_total = vap_peque['Caudal Aéreo -5°C'].values[0] * n_peque
q_nat_total = q_nat_grande + q_nat_peque_total

print(f"Vaporización natural {dep_grande_ref}: {q_nat_grande} kg/h")
print(f"Vaporización natural {dep_peque_ref} ({n_peque} uds): {q_nat_peque_total:.1f} kg/h")
print(f"Vaporización natural total (-5ºC, 2 bar, 20% llenado): {q_nat_total:.1f} kg/h")

if q_nat_total < caudal_nec_kgh:
    deficit = caudal_nec_kgh - q_nat_total
    print(f"Déficit de vaporización natural: {deficit:.2f} kg/h")
    
    # 5.2. Selección de Vaporizador Interno
    # Solo se requiere en el depósito más grande para cubrir el déficit total.
    capacidades_vapi = {"VIA 150": 150, "VIA 300": 300, "VIB 500": 500}
    potencias_caldera = {"VIA 150": 17.5, "VIA 300": 35, "VIB 500": 58}
    
    vapi_seleccionado = None
    for mod, cap in capacidades_vapi.items():
        if cap >= deficit:
            vapi_seleccionado = mod
            break
    
    if vapi_seleccionado:
        print(f"\n--- Selección de Vaporización Forzada ---")
        print(f"Vaporizador Interno Seleccionado (en {dep_grande_ref.replace('LP', 'LPVI')}): {vapi_seleccionado}")
        print(f"Capacidad forzada: {capacidades_vapi[vapi_seleccionado]} kg/h")
        print(f"Potencia de caldera requerida: {potencias_caldera[vapi_seleccionado]} kW")
        print(f"Vaporización Total (Nat + Forz): {q_nat_total + capacidades_vapi[vapi_seleccionado]:.2f} kg/h")
        print(f"RESULTADO: Suministro garantizado mediante sistema mixto (Vaporizador en el depósito de {vol_grande_m3}m3).")
    else:
        print("No se encontró un vaporizador interno suficiente.")
else:
    print("Vaporización natural suficiente.")

Demanda punta necesaria: 187.81 kg/h
Vaporización natural LP46A-22: 47.0 kg/h
Vaporización natural LP26A-22 (3 uds): 84.3 kg/h
Vaporización natural total (-5ºC, 2 bar, 20% llenado): 131.3 kg/h
Déficit de vaporización natural: 56.51 kg/h

--- Selección de Vaporización Forzada ---
Vaporizador Interno Seleccionado (en LPVI46A-22 - MODELO ESPECIAL): VIA 150
Capacidad forzada: 150 kg/h
Potencia de caldera requerida: 17.5 kW
Vaporización Total (Nat + Forz): 281.30 kg/h
RESULTADO: Suministro garantizado mediante sistema mixto (Modelo especial por limitación de parcela).
